# 1 · Bid request and the data behind it

<img src="https://redis.io/wp-content/uploads/2024/04/Logotype.svg?auto=webp&quality=85,75&width=120" alt="Redis"/>

<a href="https://colab.research.google.com/github/redis-field-engineering/redis-dsp-demo/blob/main/notebooks/01_bid_request_and_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Part of the Redis DSP candidate-generation demo. The full sequence walks the bid path from naive to fast across six notebooks; this is `01_bid_request_and_data.ipynb`.

**To run in Colab:** click the badge above, then *Runtime → Run all*. The setup cells below clone the repo, install dependencies, start a Redis Stack server, and load the synthetic dataset.

**To run locally:** make sure the docker-compose stack is up (`make up` from the repo root). The setup cells detect a local environment and skip the Colab-specific steps.

## Setup

These five cells prepare the environment. They are idempotent — safe to re-run, safe in either Colab or local. On Colab the first run takes about 60–90 seconds (pip install + apt install + dataset generation). Subsequent runs are near-instant because everything is cached.

In [1]:
# Setup 1/5 · clone the repo (Colab only).
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.path.exists("pyproject.toml"):
    print("Cloning https://github.com/redis-field-engineering/redis-dsp-demo ...")
    os.system("git clone -q https://github.com/redis-field-engineering/redis-dsp-demo.git _repo")
    os.system("cp -R _repo/. ./")
    os.system("rm -rf _repo")
    print("Repo cloned.")
elif not IN_COLAB:
    print("Local environment detected — skipping clone.")
else:
    print("Repo already present.")

Local environment detected — skipping clone.


In [2]:
# Setup 2/5 · install Python dependencies (Colab only).
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    os.system(
        f'{sys.executable} -m pip install -q '
        '"redis[hiredis]>=5.2.0" "pydantic>=2.9.0" "pandas>=2.2.0" "pyarrow>=18.0.0"'
    )
    print("Dependencies installed.")
else:
    print("Local environment detected — skipping pip install (assumes deps are already installed).")

Local environment detected — skipping pip install (assumes deps are already installed).


In [3]:
# Setup 3/5 · install and start Redis Stack (Colab only).
import os, sys, shutil
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if shutil.which("redis-stack-server") is None:
        print("Installing redis-stack-server ...")
        os.system(
            'curl -fsSL https://packages.redis.io/gpg | '
            'sudo gpg --dearmor -o /usr/share/keyrings/redis-archive-keyring.gpg'
        )
        os.system(
            'echo "deb [signed-by=/usr/share/keyrings/redis-archive-keyring.gpg] '
            'https://packages.redis.io/deb $(lsb_release -cs) main" '
            '| sudo tee /etc/apt/sources.list.d/redis.list > /dev/null'
        )
        os.system("sudo apt-get update -qq > /dev/null 2>&1")
        os.system("sudo apt-get install -qq -y redis-stack-server > /dev/null 2>&1")
    os.system("redis-stack-server --daemonize yes > /dev/null 2>&1")
    print("redis-stack-server started on :6379")
else:
    print("Local environment detected — skipping Redis install (expecting docker-compose Redis at localhost:6381).")

Local environment detected — skipping Redis install (expecting docker-compose Redis at localhost:6381).


In [4]:
# Setup 4/5 · choose the Redis URL.
import os, sys
IN_COLAB = "google.colab" in sys.modules
default_port = "6379" if IN_COLAB else "6381"
REDIS_HOST = os.getenv("REDIS_HOST", "localhost")
REDIS_PORT = os.getenv("REDIS_PORT", default_port)
REDIS_PASSWORD = os.getenv("REDIS_PASSWORD", "")
auth = f":{REDIS_PASSWORD}@" if REDIS_PASSWORD else ""
REDIS_URL = f"redis://{auth}{REDIS_HOST}:{REDIS_PORT}/0"
os.environ["DEMO_REDIS_URL"] = REDIS_URL
print(f"Redis URL: {REDIS_URL}")

Redis URL: redis://localhost:6381/0


In [5]:
# Setup 5/5 · generate and load the synthetic dataset (only if Redis is empty).
import sys, subprocess
from pathlib import Path

# Find the repo root so we can run `python -m data.synthetic` reliably.
_repo_root = Path.cwd().resolve()
while _repo_root != _repo_root.parent and not (_repo_root / "pyproject.toml").exists():
    _repo_root = _repo_root.parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from redis import Redis
client = Redis.from_url(REDIS_URL, decode_responses=True)
if not client.ping():
    raise RuntimeError(f"Redis at {REDIS_URL} did not answer PING")

if client.exists("meta:dataset_loaded"):
    print(
        f"Dataset already loaded: "
        f"{client.get('meta:user_count')} users, "
        f"{client.get('meta:campaign_count')} campaigns."
    )
else:
    print("Generating synthetic dataset (~30 seconds) ...")
    subprocess.run(
        [sys.executable, "-m", "data.synthetic",
         "--output", "data/generated/synthetic",
         "--num-users", "4000",
         "--num-campaigns", "2500",
         "--num-interactions", "120000",
         "--feature-count", "12"],
        cwd=_repo_root, check=True,
    )
    print("Loading dataset into Redis ...")
    subprocess.run(
        [sys.executable, "-m", "data.load_redis",
         "--redis-url", REDIS_URL,
         "--dataset-dir", "data/generated/synthetic"],
        cwd=_repo_root, check=True,
    )
    print(
        f"Done. {client.get('meta:user_count')} users, "
        f"{client.get('meta:campaign_count')} campaigns."
    )

Dataset already loaded: 4000 users, 2500 campaigns.


## Walkthrough

From here on the notebook is the demonstration.

In [6]:
# Locate the repo root so `notebooks._demo_setup` is importable regardless
# of where the kernel was launched (the package layout requires the repo
# root on sys.path).
import sys
from pathlib import Path
_repo_root = Path.cwd().resolve()
while _repo_root != _repo_root.parent and not (_repo_root / "pyproject.toml").exists():
    _repo_root = _repo_root.parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from notebooks._demo_setup import connect_redis
client = connect_redis()

connected to redis://localhost:6381/0
  users=4000  campaigns=2500  precompute_version=v17_2500_12


## A single MAID profile

`maid:{maid_id}` is a Redis HASH carrying the cardholder's full profile.
At the customer scale this is up to 500 M entries with up to 500 taxonomy
labels and 50 publisher-hashed identities each; the synthetic dataset uses
a coarsened version of the same shape so the mechanics are identical.


In [7]:
raw = client.hgetall('maid:maid_00042')
print('hash field count:', len(raw))
print('sample fields:')
for key in ['user_id', 'geo', 'state', 'device', 'card_tier']:
    print(f'  {key:>18} = {raw[key]!r}')
print(f'  {"interests_json":>18} = {raw["interests_json"][:120]} ...')

hash field count: 14
sample fields:
             user_id = 'maid_00042'
                 geo = 'US'
               state = 'IL'
              device = 'Roku'
           card_tier = 'Standard'
      interests_json = {"camping": 0.3801, "family": 0.2108, "finance": 0.6832, "fitness": 0.8003, "foodie": 0.3884, "gaming": 0.615, "home_imp ...


## The hot scoring profile

`maid_hot:{maid_id}` is a stripped-down hash carrying only the fields the
*online* path needs. Most of the cold profile (state, postal_code, full
identity list) is left in `maid:` for offline jobs and audit. This split
matters once the dataset is at production scale — the bid path only ever
reads the hot subset, which fits in a single `HMGET` round trip.


In [8]:
hot = client.hmget('maid_hot:maid_00042', 'user_id', 'interests_json', 'impression_count')
print('user_id          =', hot[0])
print('impression_count =', hot[2])
print('interests_json   =', hot[1][:160], '...')

user_id          = maid_00042
impression_count = 233
interests_json   = {"camping": 0.3801, "family": 0.2108, "finance": 0.6832, "fitness": 0.8003, "foodie": 0.3884, "gaming": 0.615, "home_improvement": 0.1571, "luxury": 0.18, "pet_ ...


## Identity resolution

The publisher hands the bid engine a tokenised identifier. Bid time, the
engine resolves it to a `maid_id` via a single `GET` against an `identity:`
keyspace populated alongside the MAIDs.


In [9]:
sample_token = 'id_00042_01'
maid_id = client.get(f'identity:{sample_token}')
print(f'identity:{sample_token}  ->  {maid_id}')

identity:id_00042_01  ->  maid_00042


## Active ad cache

`campaign:{id}` and `campaign_state:{id}` together carry an ad's static
targeting and its mutable delivery state (pacing, budget, frequency cap).
Static fields update every few minutes from the ad-platform feed; mutable
state updates in real time as wins land.


In [10]:
campaign = client.hgetall('campaign:c00042')
state = client.hgetall('campaign_state:c00042')

print('campaign:c00042 (static):')
for key in ['campaign_id', 'geo_json', 'device_json', 'card_tiers_json',
            'required_segments_json', 'any_of_segments_json',
            'taxonomy_filter_json', 'bid']:
    if key in campaign:
        print(f'  {key:>22} = {campaign[key][:80]}')
print()
print('campaign_state:c00042 (mutable):')
for key, value in state.items():
    print(f'  {key:>22} = {value}')

campaign:c00042 (static):
             campaign_id = c00042
                geo_json = ["*"]
             device_json = ["Web"]
         card_tiers_json = ["*"]
  required_segments_json = ["fitness_high", "camping_high"]
    any_of_segments_json = []
    taxonomy_filter_json = {"and": [{"gte": ["tech", 0.55]}, {"gte": ["foodie", 0.35]}, {"not": {"or": [{"g
                     bid = 3.724

campaign_state:c00042 (mutable):
             campaign_id = c00042
           pacing_status = active
        daily_budget_usd = 8933.91
         spent_today_usd = 2679.2
           frequency_cap = 2
                  status = active


## What the bid path needs from this data

For a single bid, the engine needs to:

1. resolve `identity_token` → `maid_id` (one Redis op),
2. fetch the MAID's scoring fields (one Redis op),
3. filter the ~5 K active ads against the MAID's static profile, mutable
   state, and the per-ad `taxonomy_filter`,
4. score the survivors and return the top few.

The next five notebooks are seven different ways to do step 3, in order
from naive to fast.


In [11]:
# Brief sanity check: do we have all the keyspaces we'll need across the demo?
for prefix in ['maid:', 'maid_hot:', 'identity:', 'aud:', 'campaign:',
                'campaign_state:', 'idx:', 'fcap:', 'bm:']:
    cursor, keys = client.scan(cursor=0, match=f'{prefix}*', count=200)
    print(f'  {prefix:<18} sample: {keys[:2]}')

  maid:              sample: ['maid:maid_00988', 'maid:maid_03663']
  maid_hot:          sample: ['maid_hot:maid_02361', 'maid_hot:maid_02670']
  identity:          sample: ['identity:id_02570_04', 'identity:id_00420_01']
  aud:               sample: ['aud:maid_01342', 'aud:maid_03201']
  campaign:          sample: ['campaign:c01923', 'campaign:c00039']
  campaign_state:    sample: ['campaign_state:c01265', 'campaign_state:c01559']
  idx:               sample: ['idx:geo:US']
  fcap:              sample: ['fcap:maid_03553', 'fcap:maid_00859']
  bm:                sample: []
